# Precursor Intelligence: EDA and feature engineering (between the data pipeline and model building)

Works on the frozen gold dataset (`facility_month_dataset.parquet`) and its `feature_registry.yaml`. Follows DADS7305 Lecture 6 (feature engineering, transformation and selection), with Lectures 4, 5 and 7 (coverage, skew/drift, schema/lineage). See `FEATURE_ENGINEERING_EDA.md` for the full write-up.

| Part | What |
|---|---|
| **A. Analysis** | A1 target · A2 univariate profile · A3 feature vs target (binned rates, single-feature AUC, mutual information, WOE/IV) · A4 redundancy (correlation, VIF) · A5 temporal stability (PSI, train/test skew) · A6 coverage and rare levels · A7 slices · A8 leakage scan · A9 PCA/t-SNE |
| **B. Feature engineering** | B1 transforms (log, clip, per-bed) · B2 bins · B3 categorical vocab + target encoding · B4 crosses · B5 peer/within-state ranks · B6 trends and momentum · B7 missingness · B8 PCA on quality measures · B9 selection (filter, embedded, permutation, stability) |
| **C. Check** | Baselines vs v1 vs v2 on the test month; save the fitted transformer, `feature_set_v2.yaml`, decisions log, and reports |

**Rules:** every fitted parameter (clip points, bins, vocabularies, encodings, PCA) is learned on the **development rows only** (DEV = train + calib splits), then applied to every month (Lecture 6: full-pass transforms, training/serving consistency). The test month is touched only in part C, as a reported final check: the v1-vs-v2 decision is made on a DEV holdout of facilities.

In [1]:
# ---- Setup ----
import os, json, warnings, math
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")

SAVE      = "/content/drive/MyDrive/nursing dataset/surveyhorizon"          # where the frozen dataset + configs live
DATA_PATH = f"{SAVE}/data/gold/facility_month_dataset.parquet"             # or output/dataset/facility_month_dataset.parquet
REGISTRY  = f"{SAVE}/configs/feature_registry.yaml"
OUT       = f"{SAVE}/feature_engineering"                                    # reports, figures, artifacts
LABEL     = "y_g12"                                                         # main label for analysis
SEED      = 42
# local runs: override any path with environment variables (FE_SAVE, FE_DATA_PATH, FE_REGISTRY, FE_OUT); .csv, .parquet and .pkl all load
SAVE, DATA_PATH, REGISTRY, OUT = (os.environ.get(f"FE_{k}", v) for k, v in [("SAVE", SAVE), ("DATA_PATH", DATA_PATH), ("REGISTRY", REGISTRY), ("OUT", OUT)])

import numpy as np, pandas as pd, yaml, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
for d in ("reports", "figures", "artifacts"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{OUT}/figures/{name}.png", dpi=120); plt.show()
print("output folder:", OUT)

output folder: C:/Users/WINDOW~1/AppData/Local/Temp/claude/C--Users-Windows-11-Desktop-NEU-sem3-IE7275-MLOPS-project/daac9cc8-fc5a-4dbf-b177-95fe6df39b58/scratchpad/fe_run/out


In [2]:
# ---- Load the frozen dataset + registry ----
ds = (pd.read_csv(DATA_PATH, dtype={"ccn": str, "t": str}, low_memory=False) if DATA_PATH.endswith(".csv") else
      pd.read_pickle(DATA_PATH) if DATA_PATH.endswith(".pkl") else pd.read_parquet(DATA_PATH))
ds["t"] = ds["t"].astype(str)
ds = ds.sort_values(["ccn", "t"]).reset_index(drop=True)
_reg = yaml.safe_load(open(REGISTRY))["features"]
reg = _reg if isinstance(_reg, dict) else {f["name"]: f for f in _reg}   # registry lists features as [{name: ..., ...}]
K = LABEL.replace("y_", "")
STATUS, SPLIT = f"status_{K}", f"split_{K}"
MODEL_FEATS = [c for c, v in reg.items() if v["model_use"] == "model" and c in ds.columns]
LABEL_LIKE = [c for c in ds.columns if c.startswith(("y_", "status_", "split_")) or c.endswith("_fwd") or c in ("exited_within_12m",)]

def as_num(s):
    """numeric view: numbers and booleans as float; True/False objects mapped; None if categorical"""
    if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").astype(float)
    vals = set(s.dropna().unique().tolist())
    if vals and vals <= {True, False}:
        return s.map({True: 1.0, False: 0.0}).astype(float)
    return None
NUM = [c for c in MODEL_FEATS if as_num(ds[c]) is not None]
CAT = [c for c in MODEL_FEATS if c not in NUM]

# labeled = final outcome known; "score" months are excluded (their positives are known early but negatives are not yet)
labeled = ds[ds[STATUS].isin(["positive", "negative"]) & (ds[SPLIT] != "score")].copy()
DEV  = labeled[labeled[SPLIT].isin(["train", "calib"])].copy()       # all fitting / analysis that informs modeling
TEST = labeled[labeled[SPLIT] == "test"].copy()                       # used only in part C
y_dev = DEV[LABEL].astype(int).values
print(f"rows {len(ds):,} | months {ds['t'].min()} → {ds['t'].max()} | model features {len(MODEL_FEATS)} ({len(NUM)} numeric, {len(CAT)} categorical)")
print(f"labeled {len(labeled):,} | DEV {len(DEV):,} rows (months {sorted(DEV['t'].unique())[:3]}…) | TEST {len(TEST):,} rows (months {sorted(TEST['t'].unique())})")

rows 222,677 | months 2023-01 → 2026-08 | model features 133 (101 numeric, 32 categorical)
labeled 117,782 | DEV 44,295 rows (months ['2023-01', '2023-04', '2023-07']…) | TEST 14,665 rows (months ['2024-10'])


# Part A: Analysis

## A1. Target: base rates, imbalance, status mix

In [3]:
status_mix = ds.groupby("t")[STATUS].value_counts().unstack(fill_value=0)
rate_t = labeled.groupby("t")[LABEL].mean()
split_rate = labeled.groupby(SPLIT)[LABEL].agg(["size", "mean"]).rename(columns={"size": "rows", "mean": "base_rate"})
display(split_rate.round(3)); display(status_mix.tail(6))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
rate_t.plot(ax=ax[0], marker="o"); ax[0].set_title(f"{LABEL}: base rate by month (labeled rows)"); ax[0].set_ylabel("share positive")
status_mix.plot.area(ax=ax[1], alpha=.7); ax[1].set_title("label status by month")
savefig("A1_target_by_month")
st = labeled.groupby("state")[LABEL].agg(["size", "mean"]).query("size >= 200").sort_values("mean")
print(f"imbalance: {labeled[LABEL].mean():.1%} positive overall; states (≥200 rows) range {st['mean'].min():.1%} → {st['mean'].max():.1%}")
pd.concat([split_rate.reset_index(), st.reset_index()], axis=1).to_csv(f"{OUT}/reports/A1_target.csv", index=False)

,rows,base_rate
split_g12,,
calib,14746,0.273
purge,58822,0.275
test,14665,0.265
train,29549,0.259


status_g12,censored,immature,negative,positive
t,,,,
2025-04,0,14769,0,0
2025-07,0,14752,0,0
2025-10,0,14721,0,0
2025-12,0,14713,0,0
2026-04,0,14699,0,0
2026-08,0,14690,0,0


imbalance: 27.0% positive overall; states (≥200 rows) range 6.5% → 62.9%


## A2. Univariate profile (distribution, zeros, outliers, missingness, cardinality)

In [4]:
rows = []
for c in MODEL_FEATS:
    s = DEV[c]; x = as_num(s)
    if x is not None:
        q = x.quantile([.01, .5, .99])
        rows.append(dict(feature=c, family=reg[c]["family"], kind="numeric", nonnull=x.notna().mean(), zero_share=(x == 0).mean(),
                         mean=x.mean(), std=x.std(), skew=x.skew(), p01=q.iloc[0], p50=q.iloc[1], p99=q.iloc[2], max=x.max(), n_unique=x.nunique()))
    else:
        vc = s.astype(str).value_counts(normalize=True)
        rows.append(dict(feature=c, family=reg[c]["family"], kind="categorical", nonnull=s.notna().mean(), n_unique=s.nunique(),
                         top_level=vc.index[0], top_share=vc.iloc[0]))
prof = pd.DataFrame(rows)
prof.to_csv(f"{OUT}/reports/A2_univariate_profile.csv", index=False)
display(prof[prof.kind == "numeric"].sort_values("skew", ascending=False).head(12).round(3))
display(prof[prof.kind == "categorical"].round(3))
skewed = prof[(prof["kind"] == "numeric") & (prof["skew"].abs() > 2) & (prof["n_unique"] > 10)]["feature"].tolist()
print(f"{len(skewed)} heavily skewed numeric features (|skew| > 2): {skewed[:15]}")
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, c in zip(axes.ravel(), skewed[:8]):
    x = as_num(DEV[c]).dropna(); ax.hist(x.clip(upper=x.quantile(.995)), bins=40); ax.set_title(c, fontsize=9)
savefig("A2_skewed_histograms")
miss = prof.set_index("feature")["nonnull"].sort_values().head(20)
miss.plot.barh(figsize=(8, 6), title="lowest non-null share (DEV)"); savefig("A2_missingness")

,feature,family,kind,nonnull,zero_share,mean,std,skew,p01,p50,p99,max,n_unique,top_level,top_share
109,qm_ls_restraints,quality,numeric,0.976,0.913,0.139,1.402,43.479,0.000,0.000,3.351,100.000,2185,NaN,NaN
39,denial_days_24m,enforcement,numeric,1.000,0.889,4.098,18.752,10.380,0.000,0.000,91.060,741.000,198,NaN,NaN
34,n_uncorrected,citations,numeric,1.000,0.908,0.375,1.882,8.348,0.000,0.000,10.000,49.000,37,NaN,NaN
35,n_fines_24m,enforcement,numeric,1.000,0.396,1.565,3.004,7.757,0.000,1.000,13.000,73.000,61,NaN,NaN
37,max_fine_24m,enforcement,numeric,1.000,0.396,17291.728,44262.156,6.838,0.000,983.000,224378.260,2103938.000,4727,NaN,NaN
54,changed_ownership_12m,facility_staffing,numeric,1.000,0.974,0.026,0.158,6.016,0.000,0.000,1.000,1.000,2,NaN,NaN
62,hprd_pt,facility_staffing,numeric,0.974,0.065,0.070,0.084,5.669,0.000,0.052,0.386,2.467,16167,NaN,NaN
61,hprd_weekend_rn,facility_staffing,numeric,0.974,0.002,0.442,0.388,5.552,0.062,0.355,1.862,9.036,32698,NaN,NaN
17,sev_points_per_bed_24m,citations,numeric,1.000,0.085,1.043,1.504,5.460,0.000,0.590,6.815,39.556,8282,NaN,NaN
14,n_cit_JL_24m,citations,numeric,1.000,0.835,0.305,0.920,5.400,0.000,0.000,4.000,18.000,19,NaN,NaN


,feature,family,kind,nonnull,zero_share,mean,std,skew,p01,p50,p99,max,n_unique,top_level,top_share
44,ownership_group,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,for_profit,0.714
45,provider_type,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,Medicare and Medicaid,0.945
46,state,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53,TX,0.080
47,resident_family_council,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,RESIDENT,0.807
48,sprinkler_status,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,YES,0.998
52,sff_status,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NONE,0.966
70,turnover_total_fn,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,none,0.847
71,turnover_rn_fn,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,none,0.781
72,admin_departures_fn,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,none,0.708
73,reported_staffing_fn,facility_staffing,categorical,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,none,0.974


41 heavily skewed numeric features (|skew| > 2): ['n_complaint_surveys_cited_12m', 'n_complaint_surveys_cited_24m', 'n_ic_surveys_12m', 'n_cit_12m', 'n_cit_24m', 'n_cit_AC_24m', 'n_cit_DF_24m', 'n_cit_GI_24m', 'n_cit_JL_24m', 'sev_points_12m', 'sev_points_24m', 'sev_points_per_bed_24m', 'n_gplus_surveys_36m', 'n_complaint_cit_12m', 'n_complaint_gplus_24m']


## A3. Feature vs target: binned rates, single-feature AUC, mutual information, WOE/IV

In [5]:
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import GroupKFold

A3_FOLDS = list(GroupKFold(n_splits=5).split(DEV, y_dev, groups=DEV["ccn"]))
def oof_rate(s):
    """categorical → out-of-fold label rate (folds grouped by facility), so a level never scores its own rows"""
    s = s.astype(str); ys = pd.Series(y_dev, index=s.index); out = pd.Series(np.nan, index=s.index)
    for tr, va in A3_FOLDS:
        out.iloc[va] = s.iloc[va].map(ys.iloc[tr].groupby(s.iloc[tr]).mean()).fillna(ys.iloc[tr].mean()).values
    return out

def woe_iv(x, y, bins=10):
    """WOE/IV with decile bins (+ a missing bin); smoothing 0.5. IV < .02 weak, .02-.1 useful, .1-.3 medium, > .3 strong (check leakage)."""
    x = pd.Series(x)
    b = pd.qcut(x.rank(method="first"), bins, labels=False) if x.notna().sum() > bins and x.nunique() > bins else x
    b = b.astype(object).where(x.notna(), "missing")
    t = pd.DataFrame({"b": b.astype(str), "y": y}).groupby("b")["y"].agg(["sum", "count"])
    good, bad = (t["count"] - t["sum"] + .5), (t["sum"] + .5)
    pg, pb = good / good.sum(), bad / bad.sum()
    return float(((pg - pb) * np.log(pg / pb)).sum())

out = []
for c in MODEL_FEATS:
    x = as_num(DEV[c])
    if x is None:                                     # categorical: out-of-fold rate encoding, only to rank
        x = oof_rate(DEV[c])
    xx = x.fillna(x.min() - 1 if x.notna().any() else 0)
    try:
        auc = roc_auc_score(y_dev, xx); ap = max(average_precision_score(y_dev, xx), average_precision_score(y_dev, -xx))
    except ValueError:
        auc, ap = np.nan, np.nan
    out.append(dict(feature=c, auc=max(auc, 1 - auc), pr_auc=ap, iv=woe_iv(x, y_dev), direction="+" if auc >= .5 else "−"))
sig = pd.DataFrame(out)
samp = DEV.sample(min(len(DEV), 20000), random_state=SEED)
Xmi = pd.DataFrame({c: (as_num(samp[c]) if c in NUM else samp[c].astype("category").cat.codes) for c in MODEL_FEATS})
Xmi = Xmi.fillna(Xmi.median(numeric_only=True)).fillna(0)
mi = mutual_info_classif(Xmi.values, samp[LABEL].astype(int).values, discrete_features=[c in CAT for c in MODEL_FEATS], random_state=SEED)
sig["mutual_info"] = sig["feature"].map(dict(zip(MODEL_FEATS, mi)))
sig = sig.sort_values("pr_auc", ascending=False).reset_index(drop=True)
sig.to_csv(f"{OUT}/reports/A3_feature_signal.csv", index=False)
display(sig.head(20).round(4))
top = sig[sig.feature.isin(NUM)].head(8).feature.tolist()
fig, axes = plt.subplots(2, 4, figsize=(17, 6.5))
for ax, c in zip(axes.ravel(), top):
    x = as_num(DEV[c]); b = pd.qcut(x.rank(method="first"), 10, labels=False)
    r = pd.DataFrame({"b": b, "y": y_dev}).groupby("b")["y"].mean()
    ax.plot(r.index + 1, r.values, marker="o"); ax.axhline(y_dev.mean(), ls="--", c="grey"); ax.set_title(c, fontsize=9); ax.set_xlabel("decile")
savefig("A3_binned_target_rate_top8")

,feature,auc,pr_auc,iv,direction,mutual_info
0,n_cit_24m,0.6710,0.4232,0.3806,+,0.0382
1,state,0.6691,0.4204,0.3845,+,0.0422
2,sev_points_24m,0.6735,0.4189,0.3828,+,0.0369
3,n_cat_qol_care_24m,0.6606,0.4148,0.3572,+,0.0379
4,n_cit_DF_24m,0.6654,0.4134,0.3509,+,0.0351
5,state_gplus_rate_12m,0.6542,0.4102,0.3292,+,0.0462
6,n_complaint_surveys_cited_24m,0.6676,0.4083,0.3866,+,0.0339
7,n_gplus_surveys_36m,0.6467,0.3953,0.3606,+,0.0345
8,sev_points_12m,0.6399,0.3921,0.2699,+,0.0290
9,n_complaint_cit_12m,0.6421,0.3917,0.3103,+,0.0267


## A4. Redundancy: correlation clusters and VIF

In [6]:
num_top = [c for c in sig.feature if c in NUM][:40]
Xc = pd.DataFrame({c: as_num(DEV[c]) for c in num_top})
corr = Xc.corr(method="spearman")
fig, ax = plt.subplots(figsize=(13, 11)); im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7); plt.colorbar(im, fraction=.03); ax.set_title("Spearman correlation (top-40 numeric by PR-AUC)")
savefig("A4_correlation")
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(corr.columns) for b in corr.columns[i + 1:] if abs(corr.loc[a, b]) >= .85]
pairs = pd.DataFrame(pairs, columns=["feature_a", "feature_b", "spearman"]).sort_values("spearman", key=abs, ascending=False)
pairs.to_csv(f"{OUT}/reports/A4_high_correlation_pairs.csv", index=False); display(pairs.head(15).round(3))

def vif_table(X):
    X = X.fillna(X.median()); X = (X - X.mean()) / X.std().replace(0, 1)
    v = []
    for j, c in enumerate(X.columns):
        A = np.c_[np.ones(len(X)), X.drop(columns=c).values]; yv = X[c].values
        beta, *_ = np.linalg.lstsq(A, yv, rcond=None); r2 = 1 - ((yv - A @ beta) ** 2).sum() / ((yv - yv.mean()) ** 2).sum()
        v.append((c, 1 / max(1 - r2, 1e-6)))
    return pd.DataFrame(v, columns=["feature", "vif"]).sort_values("vif", ascending=False)
vif = vif_table(Xc.iloc[:, :25].sample(min(len(Xc), 20000), random_state=SEED))
vif.to_csv(f"{OUT}/reports/A4_vif.csv", index=False); display(vif.head(12).round(1))
print("VIF > 10 means strong multicollinearity: matters for the logistic baseline, not for trees.")

,feature_a,feature_b,spearman
10,denial_days_24m,n_denials_24m,0.998
7,fine_dollars_24m,max_fine_24m,0.992
2,n_cit_24m,n_cit_DF_24m,0.991
6,n_complaint_cit_12m,n_complaint_surveys_cited_12m,0.963
5,sev_points_12m,n_cit_12m,0.958
12,hprd_total,hprd_weekend_total,0.956
0,n_cit_24m,sev_points_24m,0.928
4,sev_points_24m,sev_points_per_bed_24m,0.918
13,hprd_total,adj_hprd_total,0.899
11,hprd_total,rel_hprd_total,0.896


,feature,vif
0,n_cit_24m,353.9
3,n_cit_DF_24m,252.1
1,sev_points_24m,56.4
10,rel_sev_points_24m,39.2
2,n_cat_qol_care_24m,8.0
5,n_complaint_surveys_cited_24m,7.3
12,n_complaint_surveys_cited_12m,6.8
9,n_cit_12m,6.6
17,n_complaint_gplus_24m,6.1
7,sev_points_12m,5.4


VIF > 10 means strong multicollinearity: matters for the logistic baseline, not for trees.


## A5. Temporal stability: PSI by month and train→test skew

In [7]:
def psi(ref, cur, bins=10, cat=False):
    """Population Stability Index. < .1 stable, .1-.25 moderate shift, > .25 major shift."""
    if cat:
        r = ref.astype(str).value_counts(normalize=True); c = cur.astype(str).value_counts(normalize=True)
        idx = r.index.union(c.index); r, c = r.reindex(idx, fill_value=0), c.reindex(idx, fill_value=0)
    else:
        ref, cur = ref.dropna(), cur.dropna()
        if ref.nunique() < 2 or len(cur) == 0:
            return np.nan
        edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1))); edges[0], edges[-1] = -np.inf, np.inf
        r = pd.Series(np.histogram(ref, edges)[0]) / len(ref); c = pd.Series(np.histogram(cur, edges)[0]) / len(cur)
    r, c = r + 1e-4, c + 1e-4
    return float(((c - r) * np.log(c / r)).sum())

ref_t = sorted(DEV["t"].unique())[0]
months = sorted(ds["t"].unique())
ref = ds[ds["t"] == ref_t]
P = {}
for c in MODEL_FEATS:
    isc = c in CAT
    rr = ref[c] if isc else as_num(ref[c])
    P[c] = {m: psi(rr, ds.loc[ds["t"] == m, c] if isc else as_num(ds.loc[ds["t"] == m, c]), cat=isc) for m in months}
PSI = pd.DataFrame(P).T
PSI.to_csv(f"{OUT}/reports/A5_psi_by_month.csv")
worst = PSI.max(axis=1).sort_values(ascending=False)
display(worst.head(15).round(3).to_frame("max PSI vs " + ref_t))
fig, ax = plt.subplots(figsize=(14, 6)); hm = PSI.loc[worst.index[:30]]
im = ax.imshow(hm.values.astype(float), aspect="auto", cmap="magma_r", vmin=0, vmax=.5)
ax.set_yticks(range(len(hm))); ax.set_yticklabels(hm.index, fontsize=7); ax.set_xticks(range(len(months))); ax.set_xticklabels(months, rotation=90, fontsize=7)
plt.colorbar(im, fraction=.02); ax.set_title(f"PSI vs {ref_t} (30 least stable features)"); savefig("A5_psi_heatmap")
unstable = worst[worst > .25].index.tolist()
print(f"{len(unstable)} features with PSI > 0.25 in some month (check whether it is real change or a CMS methodology change): {unstable[:15]}")

,max PSI vs 2023-01
era,18.421
qm_ls_incontinence,5.338
admin_departures_fn,3.426
turnover_rn_fn,2.774
n_ic_surveys_12m,2.378
turnover_total_fn,1.798
state_median_hprd_total,1.735
max_fine_24m,1.594
fine_dollars_24m,1.301
state_gplus_rate_12m,1.257


22 features with PSI > 0.25 in some month (check whether it is real change or a CMS methodology change): ['era', 'qm_ls_incontinence', 'admin_departures_fn', 'turnover_rn_fn', 'n_ic_surveys_12m', 'turnover_total_fn', 'state_median_hprd_total', 'max_fine_24m', 'fine_dollars_24m', 'state_gplus_rate_12m', 'staffing_rating_fn', 'qm_ls_pressure_ulcer', 'qm_ls_ed_per_1000', 'n_std_36m', 'n_fines_24m']


## A6. Coverage and rare levels · A7. Slices · A8. Leakage scan

In [8]:
rare = []
for c in CAT:
    vc = DEV[c].astype(str).value_counts()
    for lvl, n in vc[vc < 100].items():
        rare.append(dict(feature=c, level=lvl, rows=int(n)))
rare = pd.DataFrame(rare); rare.to_csv(f"{OUT}/reports/A6_rare_levels.csv", index=False)
print(f"rare categorical levels (< 100 DEV rows): {len(rare)}"); display(rare.head(15))

beds = as_num(labeled["certified_beds"])
slices = {"bed_band": pd.cut(beds, [0, 50, 100, 150, 200, 1e6], labels=["<50", "50-99", "100-149", "150-199", "200+"], right=False),
          "ownership_group": labeled["ownership_group"], "chain": np.where(labeled["chain_id"].notna(), "chain", "independent")}
rows = []
for name, sl in slices.items():
    g = labeled.assign(_s=np.asarray(sl)).groupby("_s")[LABEL].agg(["size", "mean"])
    rows += [dict(slice=name, value=k, rows=int(r["size"]), rate=r["mean"]) for k, r in g.iterrows()]
sl = pd.DataFrame(rows); sl.to_csv(f"{OUT}/reports/A7_slices.csv", index=False); display(sl.round(3))
print("Full bias analysis (FDR-controlled slices, rater effects) is the separate bias-slicing ticket.")

leak = sig[(sig.auc > .80) | (sig.iv > .5)]
print("A8 leakage scan: features with single-feature AUC > 0.80 or IV > 0.5 →", leak.feature.tolist() or "none (good)")
assert not set(LABEL_LIKE) & set(MODEL_FEATS), "a label-like column is registered as a model feature"

rare categorical levels (< 100 DEV rows): 18


,feature,level,rows
0,state,AK,60
1,state,DC,51
2,state,PR,18
3,state,GU,3
4,sprinkler_status,PARTIAL,88
5,qm_ls_adl_decline_fn,missing_not_submitted,48
6,qm_ls_antianxiety_fn,missing_not_submitted,48
7,qm_ls_antipsychotic_fn,missing_not_submitted,48
8,qm_ls_catheter_fn,missing_not_submitted,48
9,qm_ls_depressive_fn,missing_not_submitted,48


,slice,value,rows,rate
0,bed_band,100-149,40039,0.294
1,bed_band,150-199,12622,0.337
2,bed_band,200+,6814,0.343
3,bed_band,50-99,45120,0.246
4,bed_band,<50,13187,0.175
5,ownership_group,for_profit,84800,0.286
6,ownership_group,government,7384,0.253
7,ownership_group,non_profit,25598,0.219
8,chain,chain,57844,0.288
9,chain,independent,59938,0.252


Full bias analysis (FDR-controlled slices, rater effects) is the separate bias-slicing ticket.
A8 leakage scan: features with single-feature AUC > 0.80 or IV > 0.5 → none (good)


## A9. Visual exploration: PCA and t-SNE (sample)

In [9]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
s = DEV.sample(min(len(DEV), 3000), random_state=SEED)
Xs = pd.DataFrame({c: as_num(s[c]) for c in NUM}); Xs = Xs.fillna(Xs.median()); Xs = (Xs - Xs.mean()) / Xs.std().replace(0, 1)
Xs = Xs.loc[:, Xs.std() > 0]
pc = PCA(n_components=min(20, Xs.shape[1]), random_state=SEED).fit(Xs.values)
Z = pc.transform(Xs.values)
emb = TSNE(n_components=2, random_state=SEED, init="pca", perplexity=30).fit_transform(Z)
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].plot(np.cumsum(pc.explained_variance_ratio_), marker="o"); ax[0].set_title("PCA cumulative explained variance"); ax[0].set_xlabel("components")
for a, (xy, nm) in zip(ax[1:], [(Z[:, :2], "PCA (PC1 vs PC2)"), (emb, "t-SNE")]):
    sc = a.scatter(xy[:, 0], xy[:, 1], c=s[LABEL].astype(int), cmap="coolwarm", s=6, alpha=.6); a.set_title(f"{nm}, colored by {LABEL}")
savefig("A9_pca_tsne")

# Part B: Feature engineering
Everything below is learned on **DEV** and applied to every month by one `FeatureTransformer`, defined in the module `feature_transform_v2.py` written by the next cell. It's saved as an artifact, so monthly scoring uses exactly the same transforms (training/serving consistency).

In [10]:
%%writefile feature_transform_v2.py
"""Feature engineering v2 transformer, written by Feature_Engineering_EDA.ipynb and shipped next to its joblib.

Scoring:
    import sys, joblib; sys.path.insert(0, "<OUT>/artifacts")
    bundle = joblib.load("<OUT>/artifacts/feature_transform_v2.joblib")
    X = bundle["transformer"].transform(panel)   # panel = the month to score + each facility's earlier months (for lag3/vol4)

Every learned parameter and every column list lives on the object, so nothing depends on notebook globals.
"""
import numpy as np, pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold

LOG_CANDIDATES  = ["sev_points_12m", "sev_points_24m", "n_cit_12m", "n_cit_24m", "n_cit_DF_24m", "max_fine_24m", "denial_days_24m",
                   "n_complaint_cit_12m", "certified_beds", "days_since_last_penalty", "days_since_last_gplus"]
RANK_CANDIDATES = ["hprd_total", "hprd_rn", "turnover_total", "sev_points_24m", "n_complaint_surveys_cited_12m", "n_cit_24m", "qm_ls_falls_major_injury"]
LAG_CANDIDATES  = ["hprd_total", "n_cit_12m", "n_complaint_surveys_cited_12m", "turnover_total"]
ISNA_CANDIDATES = ["hprd_total", "turnover_total", "days_since_last_penalty", "days_since_last_gplus", "qm_ls_falls_major_injury"]
REQUIRED = ["ccn", "t", "state", "certified_beds", "n_cit_12m", "n_cit_24m", "n_complaint_surveys_cited_12m", "n_complaint_surveys_cited_24m",
            "fine_dollars_24m", "days_since_last_std", "gplus_last_std", "std_overdue", "ownership_group", "chain_id", "turnover_total", "hprd_rn"]
BED_BINS, BED_LABELS = [0, 50, 100, 150, 200, 1e6], ["<50", "50-99", "100-149", "150-199", "200+"]
STD_BINS, STD_LABELS = [-1, 180, 365, 456, 730, 1e6], ["<6m", "6-12m", "12-15m", "15-24m", "24m+"]   # inspections due at ~15 months (456 days)


def as_num(s):
    """numeric view: numbers and booleans as float; True/False objects mapped; None if categorical"""
    if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").astype(float)
    vals = set(s.dropna().unique().tolist())
    if vals and vals <= {True, False}:
        return s.map({True: 1.0, False: 0.0}).astype(float)
    return None


class FeatureTransformer:
    """Fit on DEV rows only; transform any month (Lecture 6: full-pass transforms, training/serving consistency)."""

    def __init__(self, num, cat, skewed, rare_min=30, te_smooth=50, no_vocab=("state",), seed=42):
        self.num, self.cat, self.skewed = list(num), list(cat), list(skewed)
        self.rare_min, self.te_smooth, self.no_vocab, self.seed = rare_min, te_smooth, tuple(no_vocab), seed

    def fit(self, dev, y):
        missing = [c for c in REQUIRED if c not in dev.columns]
        if missing:
            raise KeyError(f"columns the transformer needs are missing from the dataset: {missing}")
        cols = set(dev.columns)
        self.log_cols = [c for c in LOG_CANDIDATES if c in cols]
        self.clip_cols = [c for c in self.num if c in self.skewed or c in self.log_cols]
        self.rank_cols = [c for c in RANK_CANDIDATES if c in cols]
        self.lag_cols = [c for c in LAG_CANDIDATES if c in cols]
        self.isna_cols = [c for c in ISNA_CANDIDATES if c in cols]
        self.qm_cols = [c for c in self.num if c.startswith(("qm_ls_", "qm_ss_")) and as_num(dev[c]).notna().any()]
        self.vocab_cols = [c for c in self.cat if c not in self.no_vocab]      # raw state never gets a _v copy (bias mitigation)
        self.clip = {c: tuple(as_num(dev[c]).quantile([.01, .99])) for c in self.clip_cols}
        self.vocab = {c: set(dev[c].astype(str).value_counts().loc[lambda v: v >= self.rare_min].index) for c in self.vocab_cols}
        self.prior = float(np.mean(y))
        self.te_state = self._te(dev["state"], y, self.prior)
        Q = self._qm_matrix(dev)
        self.qm_med = Q.median(); Qf = Q.fillna(self.qm_med)
        self.qm_mu, self.qm_sd = Qf.mean(), Qf.std().replace(0, 1)
        self.pca = PCA(n_components=min(3, Q.shape[1]), random_state=self.seed).fit(((Qf - self.qm_mu) / self.qm_sd).values) if Q.shape[1] else None
        return self

    def _te(self, s, y, prior):
        g = pd.DataFrame({"k": s.astype(str).values, "y": np.asarray(y)}).groupby("k")["y"].agg(["sum", "count"])
        return ((g["sum"] + self.te_smooth * prior) / (g["count"] + self.te_smooth)).to_dict()

    def oof_state_encoding(self, dev, y, groups, n_splits=5):
        """state_te for the DEV rows themselves: out-of-fold, folds grouped by facility, so a home never sees its own label."""
        y = np.asarray(y); out = pd.Series(np.nan, index=dev.index)
        for tr, va in GroupKFold(n_splits=n_splits).split(dev, y, groups=groups):
            prior = float(y[tr].mean()); te = self._te(dev["state"].iloc[tr], y[tr], prior)
            out.iloc[va] = dev["state"].iloc[va].astype(str).map(te).fillna(prior).values
        return out

    def _qm_matrix(self, d):
        return pd.DataFrame({c: as_num(d[c]) for c in self.qm_cols}, index=d.index)

    def _panel_lags(self, d):
        """B6 lags: change vs three snapshots earlier and 4-snapshot volatility, from the same facility's earlier rows only."""
        f = d.sort_values(["ccn", "t"]); out = pd.DataFrame(index=f.index)
        for c in self.lag_cols:
            v = as_num(f[c]); g = v.groupby(f["ccn"])
            out[f"lag3_diff_{c}"] = v - g.shift(3)
            out[f"vol4_{c}"] = g.transform(lambda s: s.rolling(4, min_periods=3).std())
        return out.reindex(d.index)

    def transform(self, d):
        X = pd.DataFrame(index=d.index)
        peer = [d["t"], d["state"]]
        # B1 clip + log + per-bed normalisation
        for c in self.clip_cols:
            lo, hi = self.clip[c]; X[f"{c}_clip"] = as_num(d[c]).clip(lo, hi)
        for c in self.log_cols:
            X[f"log_{c}"] = np.log1p(as_num(d[c]).clip(lower=0))
        beds = as_num(d["certified_beds"]).where(lambda b: b > 0)
        compl12 = as_num(d["n_complaint_surveys_cited_12m"])
        X["cit_per_bed_24m"] = as_num(d["n_cit_24m"]) / beds
        X["complaints_per_100beds_12m"] = 100 * compl12 / beds
        X["fine_dollars_per_bed_24m"] = np.expm1(as_num(d["fine_dollars_24m"])) / beds      # fine_dollars_24m is stored as log1p
        # B2 bins
        X["bed_band"] = pd.cut(as_num(d["certified_beds"]), BED_BINS, labels=BED_LABELS, right=False).astype(str)
        X["std_recency_band"] = pd.cut(as_num(d["days_since_last_std"]), STD_BINS, labels=STD_LABELS).astype(str)
        # B3 rare-level grouping + target encoding of state (the notebook overwrites DEV rows with the out-of-fold version)
        for c in self.vocab_cols:
            s = d[c].astype(str); X[f"{c}_v"] = s.where(s.isin(self.vocab[c]), "other")
        X["state_te"] = d["state"].astype(str).map(self.te_state).fillna(self.prior)
        # B4 crosses
        rn_rank = as_num(d["hprd_rn"]).groupby(peer).rank(pct=True)
        X["x_complaints_x_low_rn"] = compl12 * (1 - rn_rank)          # complaints × RN staffing below state peers (same month)
        X["x_harm_last_std_x_overdue"] = as_num(d["gplus_last_std"]).fillna(0) * as_num(d["std_overdue"]).fillna(0)
        X["x_size_ownership"] = X["bed_band"] + "|" + d["ownership_group"].astype(str)
        X["x_chain_x_turnover"] = np.where(d["chain_id"].notna(), 1.0, 0.0) * as_num(d["turnover_total"])
        # B5 peer ranks within state-month (same-month data only: no leakage)
        for c in self.rank_cols:
            X[f"pct_state_{c}"] = as_num(d[c]).groupby(peer).rank(pct=True)
        # B6 momentum (from counts already in the row) + panel lags (earlier months of the same facility only)
        X["cit_momentum_12m"] = as_num(d["n_cit_12m"]) - (as_num(d["n_cit_24m"]) - as_num(d["n_cit_12m"]))
        X["complaint_momentum_12m"] = compl12 - (as_num(d["n_complaint_surveys_cited_24m"]) - compl12)
        X = X.join(self._panel_lags(d))
        # B7 missingness indicators (where missing has meaning)
        for c in self.isna_cols:
            X[f"isna_{c}"] = d[c].isna().astype(float)
        # B8 PCA quality index
        if self.pca is not None:
            Z = self.pca.transform(((self._qm_matrix(d).fillna(self.qm_med) - self.qm_mu) / self.qm_sd).values)
            for i in range(Z.shape[1]):
                X[f"qm_pca_{i + 1}"] = Z[:, i]
        return X

Writing feature_transform_v2.py


In [11]:
# ---- Fit the transformer on DEV, apply to every month ----
import sys, shutil, importlib
sys.path.insert(0, os.getcwd())
import feature_transform_v2 as ftm; importlib.reload(ftm)
shutil.copy("feature_transform_v2.py", f"{OUT}/artifacts/feature_transform_v2.py")     # the module ships with the joblib

ft = ftm.FeatureTransformer(NUM, CAT, skewed, rare_min=30, te_smooth=50, seed=SEED).fit(DEV, y_dev)
ENG = ft.transform(ds)
# DEV rows get the out-of-fold state encoding (grouped by facility), so DEV never sees its own label
ENG.loc[DEV.index, "state_te"] = ft.oof_state_encoding(DEV, y_dev, DEV["ccn"])
NEW = list(ENG.columns)
print(f"{len(NEW)} engineered features"); display(ENG.loc[DEV.index].describe().T.head(20).round(3))

120 engineered features


,count,mean,std,min,25%,50%,75%,max
n_complaint_surveys_cited_12m_clip,44295.0,0.992,1.488,0.0,0.000,0.00,1.000,8.000
n_complaint_surveys_cited_24m_clip,44295.0,2.136,2.756,0.0,0.000,1.00,3.000,15.000
n_ic_surveys_12m_clip,44295.0,0.442,0.928,0.0,0.000,0.00,1.000,5.000
n_cit_12m_clip,44295.0,5.659,6.989,0.0,0.000,3.00,8.000,33.000
n_cit_24m_clip,44295.0,11.727,11.578,0.0,3.000,8.00,16.000,58.000
n_cit_AC_24m_clip,44295.0,0.263,0.618,0.0,0.000,0.00,0.000,3.000
n_cit_DF_24m_clip,44295.0,10.685,10.590,0.0,3.000,8.00,15.000,53.000
n_cit_GI_24m_clip,44295.0,0.455,0.978,0.0,0.000,0.00,1.000,5.000
n_cit_JL_24m_clip,44295.0,0.282,0.758,0.0,0.000,0.00,0.000,4.000
sev_points_12m_clip,44295.0,44.615,67.068,0.0,0.000,20.00,60.000,368.000


## B9. Feature selection: filter → embedded → permutation → stability
20% of DEV facilities are held out first for the v1-vs-v2 decision in part C; selection uses the other 80% (SEL).

In [12]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit
try:
    import lightgbm as lgb
    def GBM(): return lgb.LGBMClassifier(n_estimators=400, learning_rate=.05, num_leaves=31, min_child_samples=100, subsample=.8,
                                         subsample_freq=1, colsample_bytree=.8, verbose=-1, random_state=SEED)
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    def GBM(): return HistGradientBoostingClassifier(max_iter=300, learning_rate=.05, min_samples_leaf=100, categorical_features="from_dtype", random_state=SEED)

# hold out 20% of DEV facilities for the v1-vs-v2 decision in part C; selection below sees only SEL
sel_i, hold_i = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=SEED).split(DEV, y_dev, groups=DEV["ccn"]))
SEL, HOLD = DEV.index[sel_i], DEV.index[hold_i]
y_sel, y_hold = y_dev[sel_i], y_dev[hold_i]
print(f"selection rows {len(SEL):,} | decision holdout {len(HOLD):,} rows ({DEV.loc[HOLD, 'ccn'].nunique():,} facilities, {y_hold.mean():.1%} positive)")

# candidate set v2 = v1 model features (raw state replaced by its encoding) + engineered features
DROP_FROM_V1 = ["state"]                                   # bias mitigation: use state_te / peer ranks instead of the raw level
cand = [c for c in MODEL_FEATS if c not in DROP_FROM_V1] + [c for c in NEW if c not in MODEL_FEATS]
assert not {"state", "state_v"} & set(cand), "raw state must stay out of the v2 candidates (bias mitigation)"
FULL = pd.concat([ds[[c for c in MODEL_FEATS if c in ds.columns]], ENG], axis=1)
FULL = FULL.loc[:, ~FULL.columns.duplicated()]

def frame(rows_idx, cols, levels=None):
    X = pd.DataFrame(index=rows_idx); levels = {} if levels is None else levels
    for c in cols:
        n = as_num(FULL.loc[rows_idx, c])
        if n is not None:
            X[c] = n
        else:
            levels.setdefault(c, sorted(FULL.loc[DEV.index, c].astype(str).dropna().unique()))   # dropna: pandas ≥ 3 keeps NaN in astype(str)
            X[c] = pd.Categorical(FULL.loc[rows_idx, c].astype(str), categories=levels[c])
    return X, levels

log = []
# 1) filter: near-constant and near-duplicate features
Xd, LV = frame(SEL, cand)
const = [c for c in cand if (Xd[c].value_counts(normalize=True, dropna=False).iloc[0] if len(Xd[c]) else 1) > .995]
log += [dict(feature=c, step="filter", decision="drop",
             reason="all/most missing in DEV months (needs earlier history, e.g. t-6/t-12)" if Xd[c].isna().mean() > .995 else "near-constant (>99.5% one value)")
        for c in const]
cand = [c for c in cand if c not in const]
numc = [c for c in cand if pd.api.types.is_numeric_dtype(Xd[c])]
cc = Xd[numc].corr(method="spearman").abs()
def single_ap(c):
    x = Xd[c].astype(float); x = x.fillna(x.min() - 1 if x.notna().any() else 0)
    return max(average_precision_score(y_sel, x), average_precision_score(y_sel, -x))
ap = {c: single_ap(c) for c in numc}
dup = set()
for i, a in enumerate(numc):
    for b in numc[i + 1:]:
        if a not in dup and b not in dup and cc.loc[a, b] >= .95:
            loser = a if ap[a] < ap[b] else b; dup.add(loser)
            log.append(dict(feature=loser, step="filter", decision="drop", reason=f"|ρ|≥.95 with {b if loser == a else a} (lower single-feature PR-AUC)"))
cand = [c for c in cand if c not in dup]

# 2) embedded + 3) permutation importance, across 3 facility-grouped folds of SEL (stability)
imp_rows = []
for k, (tr_i, va_i) in enumerate(GroupKFold(n_splits=3).split(SEL, y_sel, groups=DEV.loc[SEL, "ccn"])):
    Xtr, lv = frame(SEL[tr_i], cand); Xva, _ = frame(SEL[va_i], cand, lv)
    m = GBM().fit(Xtr, y_sel[tr_i])
    pi = permutation_importance(m, Xva, y_sel[va_i], scoring="average_precision", n_repeats=3, random_state=SEED, n_jobs=-1)
    gain = m.booster_.feature_importance("gain") if hasattr(m, "booster_") else np.full(len(cand), np.nan)
    imp_rows += [dict(fold=k, feature=c, perm=pi.importances_mean[j], gain=gain[j]) for j, c in enumerate(cand)]
IMP = pd.DataFrame(imp_rows)
stab = IMP.groupby("feature").agg(perm_mean=("perm", "mean"), perm_min=("perm", "min"), folds_positive=("perm", lambda s: int((s > 0).sum())), gain_mean=("gain", "mean"))
stab = stab.sort_values("perm_mean", ascending=False)
keep = stab[(stab.folds_positive >= 2) & (stab.perm_mean > 0)].index.tolist()
for c in cand:
    if c not in keep:
        log.append(dict(feature=c, step="permutation", decision="drop", reason=f"not useful in ≥2 of 3 folds (mean Δ PR-AUC {stab.loc[c, 'perm_mean']:.4f})"))
for c in keep:
    log.append(dict(feature=c, step="final", decision="keep", reason=f"mean Δ PR-AUC {stab.loc[c, 'perm_mean']:.4f}, positive in {stab.loc[c, 'folds_positive']}/3 folds"))
FEATURE_SET_V2 = keep
LOG = pd.DataFrame(log); LOG.to_csv(f"{OUT}/reports/B9_feature_decisions_log.csv", index=False)
stab.to_csv(f"{OUT}/reports/B9_importance_stability.csv")
print(f"candidates {len(cand) + len(const) + len(dup)} → after filter {len(cand)} → selected {len(FEATURE_SET_V2)}")
stab.head(25)["perm_mean"].iloc[::-1].plot.barh(figsize=(8, 8), title="permutation importance (Δ PR-AUC), mean of 3 folds"); savefig("B9_permutation_importance")

selection rows 35,439 | decision holdout 8,856 rows (2,964 facilities, 26.4% positive)


candidates 252 → after filter 173 → selected 66


# Part C: Check against baselines on the test month, then save

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def p_at(y, s, f):
    o = np.argsort(-s, kind="stable"); return float(y[o[:max(1, int(len(y) * f))]].mean())
V1 = [c for c in MODEL_FEATS if c in FULL]

# ---- 1. Decision: v1 vs v2 on the DEV holdout (20% of facilities that B9 never saw); the test month plays no part ----
dec = {}
for name, cols in [("v1", V1), ("v2", FEATURE_SET_V2)]:
    Xtr, lv = frame(SEL, cols); Xho, _ = frame(HOLD, cols, lv)
    s = GBM().fit(Xtr, y_sel).predict_proba(Xho)[:, 1]
    dec[name] = dict(pr_auc=average_precision_score(y_hold, s), p_at_5=p_at(y_hold, s, .05))
DEC = pd.DataFrame(dec).T
ACCEPT_V2 = bool(DEC.loc["v2", "pr_auc"] > DEC.loc["v1", "pr_auc"] and DEC.loc["v2", "p_at_5"] > DEC.loc["v1", "p_at_5"])
FINAL_SET = FEATURE_SET_V2 if ACCEPT_V2 else V1
DEC.assign(accepted_v2=ACCEPT_V2).to_csv(f"{OUT}/reports/C_v1_v2_decision_holdout.csv"); display(DEC.round(4))
print("decision:", "accept v2" if ACCEPT_V2 else "keep v1 (v2 did not beat v1 on both PR-AUC and precision@5% on the DEV holdout)")

# ---- 2. Final check on the test month: reported only, it does not change the decision ----
if len(TEST) == 0:
    print(f"no '{SPLIT}' = test rows in this dataset: skipping the test-month check; the DEV holdout decision above stands")
else:
    y_te = TEST[LABEL].astype(int).values
    def metrics(name, s):
        s = np.asarray(s, float); ok = ~np.isnan(s)
        return dict(model=name, roc_auc=roc_auc_score(y_te[ok], s[ok]), pr_auc=average_precision_score(y_te[ok], s[ok]),
                    p_at_5=p_at(y_te[ok], s[ok], .05), p_at_10=p_at(y_te[ok], s[ok], .10))
    res = []
    if "overall_rating" in TEST:
        res.append(metrics("baseline: overall star rating", -as_num(TEST["overall_rating"]).fillna(3)))
    res.append(metrics("baseline: severity rule", (as_num(TEST["n_cit_GI_24m"]) + 2 * as_num(TEST["n_cit_JL_24m"]) + .1 * as_num(TEST["n_cit_24m"])).values))
    # logistic regression on engineered features (numeric: median impute + standardize; categoricals one-hot), fitted on DEV only
    def lr_frame(idx, cols, fit=None):
        num = [c for c in cols if as_num(FULL[c]) is not None]; cat = [c for c in cols if c not in num]
        Xn = pd.DataFrame({c: as_num(FULL.loc[idx, c]) for c in num})
        Xc_ = pd.get_dummies(FULL.loc[idx, cat].astype(str), dummy_na=False) if cat else pd.DataFrame(index=idx)
        if fit is None:
            fit = dict(med=Xn.median(), cols=Xc_.columns)
            Xn_ = Xn.fillna(fit["med"]); fit["sc"] = StandardScaler().fit(Xn_)
        Xn = fit["sc"].transform(Xn.fillna(fit["med"])); Xc_ = Xc_.reindex(columns=fit["cols"], fill_value=0)
        return np.c_[Xn, Xc_.values], fit
    Xl, fit = lr_frame(DEV.index, FEATURE_SET_V2); Xlt, _ = lr_frame(TEST.index, FEATURE_SET_V2, fit)
    lr = LogisticRegression(max_iter=2000, C=.5).fit(Xl, y_dev)
    res.append(metrics("logistic regression · feature set v2", lr.predict_proba(Xlt)[:, 1]))
    for name, cols in [("GBM · feature set v1 (pipeline features)", V1), ("GBM · feature set v2 (engineered + selected)", FEATURE_SET_V2)]:
        Xtr, lv = frame(DEV.index, cols); Xte, _ = frame(TEST.index, cols, lv)
        m = GBM().fit(Xtr, y_dev); res.append(metrics(name, m.predict_proba(Xte)[:, 1]))
    R = pd.DataFrame(res).set_index("model")
    R.to_csv(f"{OUT}/reports/C_baseline_comparison.csv"); display(R.round(3))
    R[["pr_auc", "p_at_5", "p_at_10"]].plot.barh(figsize=(10, 4.5), title=f"{LABEL} on the test month: baselines vs v1 vs v2")
    plt.axvline(y_te.mean(), ls="--", c="grey"); savefig("C_baseline_comparison")
    print("Note: this is a feature-set check with default settings, not the final tuned and calibrated model.")

,pr_auc,p_at_5
v1,0.5116,0.7104
v2,0.5088,0.7014


decision: keep v1 (v2 did not beat v1 on both PR-AUC and precision@5% on the DEV holdout)


,roc_auc,pr_auc,p_at_5,p_at_10
model,,,,
baseline: overall star rating,0.629,0.344,0.422,0.389
baseline: severity rule,0.663,0.415,0.576,0.514
logistic regression · feature set v2,0.708,0.492,0.694,0.598
GBM · feature set v1 (pipeline features),0.706,0.482,0.649,0.579
GBM · feature set v2 (engineered + selected),0.703,0.484,0.679,0.582


Note: this is a feature-set check with default settings, not the final tuned and calibrated model.


In [14]:
# ---- Save artifacts: fitted transformer (+ its module), feature set v2, engineered dataset (optional) ----
import joblib
joblib.dump({"transformer": ft, "module": "feature_transform_v2.py", "feature_set_v2": FEATURE_SET_V2, "accepted_v2": ACCEPT_V2,
             "final_feature_set": FINAL_SET, "label": LABEL, "fitted_on_months": sorted(DEV["t"].unique())},
            f"{OUT}/artifacts/feature_transform_v2.joblib")
def describe(c):
    if c in reg: return {**reg[c], "origin": "pipeline (v1)"}
    fam = ("transform" if c.endswith("_clip") or c.startswith("log_") else "binning" if c in ("bed_band", "std_recency_band") else
           "categorical" if c.endswith("_v") or c == "state_te" else "cross" if c.startswith("x_") else "peer rank" if c.startswith("pct_state_") else
           "trend" if c.startswith(("lag3_", "vol4_")) or "momentum" in c else "missingness" if c.startswith("isna_") else "pca" if c.startswith("qm_pca") else "derived")
    return {"family": f"engineered: {fam}", "model_use": "model", "origin": "feature_engineering_v2", "fit_on": "DEV only"}
fs = {"version": 2, "label": LABEL, "fitted_on_months": sorted(DEV["t"].unique()), "n_features": len(FEATURE_SET_V2),
      "accepted": ACCEPT_V2, "decision_holdout": DEC.round(4).to_dict(orient="index"),
      "dropped_from_v1": DROP_FROM_V1, "features": {c: describe(c) for c in FEATURE_SET_V2}}
yaml.safe_dump(fs, open(f"{OUT}/artifacts/feature_set_v2.yaml", "w"), sort_keys=False)
SAVE_DATASET = False                   # True: write the engineered dataset (large)
if SAVE_DATASET:
    out = pd.concat([ds[["ccn", "t"] + LABEL_LIKE], FULL[FEATURE_SET_V2]], axis=1)
    out.to_parquet(f"{OUT}/artifacts/facility_month_dataset_v2.parquet", index=False)
print("saved:"); [print("  ", os.path.relpath(os.path.join(r, f), OUT)) for r, _, fs_ in os.walk(OUT) for f in sorted(fs_)]

saved:
   artifacts\feature_set_v2.yaml
   artifacts\feature_transform_v2.joblib
   artifacts\feature_transform_v2.py
   figures\A1_target_by_month.png
   figures\A2_missingness.png
   figures\A2_skewed_histograms.png
   figures\A3_binned_target_rate_top8.png
   figures\A4_correlation.png
   figures\A5_psi_heatmap.png
   figures\A9_pca_tsne.png
   figures\B9_permutation_importance.png
   figures\C_baseline_comparison.png
   reports\A1_target.csv
   reports\A2_univariate_profile.csv
   reports\A3_feature_signal.csv
   reports\A4_high_correlation_pairs.csv
   reports\A4_vif.csv
   reports\A5_psi_by_month.csv
   reports\A6_rare_levels.csv
   reports\A7_slices.csv
   reports\B9_feature_decisions_log.csv
   reports\B9_importance_stability.csv
   reports\C_baseline_comparison.csv
   reports\C_v1_v2_decision_holdout.csv


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]